In [13]:
import os
import sys
import numpy as np
import pandas as pd
from stable_baselines3 import PPO

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from envs.sghaul_env import SGHaulEnv

def extrair_metricas_artigo(env, model=None, acao_estatica=None, episodios=50):
    resultados = {
        'throughput_embb': [],
        'throughput_urllc': [],
        'sla_urllc': [],
        'sla_embb': [],
        'total_falhas_embb': [],  
        'packet_loss_embb': [],   # NOVA MÉTRICA: eMBB
        'lat_media_urllc': [],
        'lat_p99_urllc': [],
        'lat_max_urllc': [],      
        'jitter_urllc': []        # NOVA MÉTRICA: URLLC
    }
    
    for ep in range(episodios):
        obs, info = env.reset(seed=(2000 + ep))
        done = False
        
        ep_th_embb, ep_th_urllc = 0.0, 0.0
        ep_viol_urllc, ep_tot_urllc = 0, 0
        ep_viol_embb, ep_tot_embb = 0, 0
        ep_latencias_urllc = []
        passos = 0
        
        while not done:
            if model is not None:
                action, _ = model.predict(obs, deterministic=True)
            else:
                action = acao_estatica
                
            obs, reward, terminated, truncated, info = env.step(action)
            
            ep_th_embb += info.get('throughput_mbps', 0.0) * 1000.0
            ep_th_urllc += info.get('throughput_urllc_mbps', 0.0) * 1000.0
            
            ep_viol_urllc += info.get('violacoes_uc16', 0)
            ep_tot_urllc += info.get('total_pacotes_uc16', 0)
            
            ep_viol_embb += info.get('violacoes_uc3', 0)
            ep_tot_embb += info.get('total_pacotes_uc3', 0)
            
            lat = info.get('avg_latency_uc16', 0.0)
            if lat > 0:
                ep_latencias_urllc.append(lat)
                
            passos += 1
            done = terminated or truncated

        resultados['throughput_embb'].append(ep_th_embb / passos if passos > 0 else 0)
        resultados['throughput_urllc'].append(ep_th_urllc / passos if passos > 0 else 0)
        
        # Guarda as falhas absolutas
        resultados['total_falhas_embb'].append(ep_viol_embb)
        
        # Calcula SLAs
        taxa_urllc = (1.0 - (ep_viol_urllc / ep_tot_urllc))*100 if ep_tot_urllc > 0 else 0.0
        taxa_embb = (1.0 - (ep_viol_embb / ep_tot_embb))*100 if ep_tot_embb > 0 else 0.0
        resultados['sla_urllc'].append(taxa_urllc)
        resultados['sla_embb'].append(taxa_embb)
        
        # NOVO: Calcula Packet Loss (%) para o eMBB
        loss_embb = (ep_viol_embb / ep_tot_embb) * 100 if ep_tot_embb > 0 else 0.0
        resultados['packet_loss_embb'].append(loss_embb)
        
        if ep_latencias_urllc:
            resultados['lat_media_urllc'].append(np.mean(ep_latencias_urllc))
            resultados['lat_p99_urllc'].append(np.percentile(ep_latencias_urllc, 99))
            resultados['lat_max_urllc'].append(np.max(ep_latencias_urllc))
            # NOVO: Calcula o Jitter (Desvio Padrão do atraso no episódio)
            resultados['jitter_urllc'].append(np.std(ep_latencias_urllc))
        else:
            resultados['lat_media_urllc'].append(np.nan)
            resultados['lat_p99_urllc'].append(np.nan)
            resultados['lat_max_urllc'].append(np.nan)
            resultados['jitter_urllc'].append(np.nan)

    # MONTAGEM FINAL DA TABELA
    tabela_final = {}
    for chave, valores in resultados.items():
        valores_validos = [v for v in valores if not np.isnan(v)]
        
        if not valores_validos:
            tabela_final[chave] = "Falha (0 pacotes)"
            continue
            
        if chave == 'total_falhas_embb':
            total_perdido = int(np.sum(valores_validos))
            tabela_final[chave] = f"{total_perdido:,}".replace(",", ".")
            
        elif chave == 'lat_max_urllc':
            tabela_final[chave] = f"{np.max(valores_validos):.2f}"
            
        else:
            tabela_final[chave] = f"{np.mean(valores_validos):.2f} ± {np.std(valores_validos):.2f}"
            
    return tabela_final

# ==========================================
# EXECUÇÃO E CRIAÇÃO DO DATAFRAME
# ==========================================
env = SGHaulEnv()
caminho_modelo = r"C:\Users\adriely\Desktop\SGHaul_DRL_Project_Testes\models\ppo_sghaul_slicing.zip"
model_ppo = PPO.load(caminho_modelo, env=env)

res_ppo = extrair_metricas_artigo(env, model=model_ppo)
res_estatico_50 = extrair_metricas_artigo(env, acao_estatica=np.array([0.5, 0.5]))
res_estatico_0 = extrair_metricas_artigo(env, acao_estatica=np.array([0.01, 0.99]))

df_tcc = pd.DataFrame([res_ppo, res_estatico_50, res_estatico_0], 
                  index=["DRL Slicing (Proposto)", "Static Slicing (50/50)", "eMBB Priority (01/99)"])

# Atualizando as colunas para incluir as duas novas métricas
df_tcc.columns = [
    "Avg Throughput eMBB (Kbps)", 
    "Avg Throughput URLLC (Kbps)", 
    "SLA Satisfaction URLLC (%)", 
    "SLA Satisfaction eMBB (%)", 
    "Total eMBB Violations (pkts)", 
    "Packet Loss eMBB (%)",         # NOVA COLUNA
    "Avg Latency URLLC (ms)", 
    "99th Percentile Latency (ms)",
    "Max Latency URLLC (ms)",
    "Jitter URLLC (ms)"             # NOVA COLUNA
]

display(df_tcc)

C:\Users\adriely\AppData\Roaming\Python\Python313\site-packages\gymnasium\spaces\box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
C:\Users\adriely\AppData\Roaming\Python\Python313\site-packages\gymnasium\spaces\box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


,Avg Throughput eMBB (Kbps),Avg Throughput URLLC (Kbps),SLA Satisfaction URLLC (%),SLA Satisfaction eMBB (%),Total eMBB Violations (pkts),Packet Loss eMBB (%),Avg Latency URLLC (ms),99th Percentile Latency (ms),Max Latency URLLC (ms),Jitter URLLC (ms)
DRL Slicing (Proposto),5.83 ± 3.67,47.45 ± 31.09,100.00 ± 0.00,88.53 ± 14.78,11.767,11.47 ± 14.78,141.11 ± 3.96,256.44 ± 4.75,270.00,77.96 ± 11.18
Static Slicing (50/50),5.14 ± 3.17,41.36 ± 26.88,100.00 ± 0.00,82.27 ± 31.39,18.428,17.73 ± 31.39,140.25 ± 3.61,250.00 ± 0.00,260.00,82.51 ± 7.93
eMBB Priority (01/99),5.33 ± 3.22,43.00 ± 27.52,100.00 ± 0.00,90.68 ± 15.73,9.714,9.32 ± 15.73,140.89 ± 3.54,250.62 ± 1.40,299.69,81.84 ± 8.36


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def plot_learning_curve_tb(tb_log_file):
    print("Gerando gráfico de Treinamento...")
    event_acc = EventAccumulator(tb_log_file)
    event_acc.Reload()
    
    scalar_tag = 'rollout/ep_rew_mean'
    events = event_acc.Scalars(scalar_tag)
    steps = [e.step for e in events]
    rewards = [e.value for e in events]
    
    plt.rcParams['font.family'] = 'serif'
    plt.figure(figsize=(7, 4)) # Mesmo tamanho dos outros padronizado!
    
    plt.plot(steps, rewards, color='purple', linewidth=2.0, label='Average Reward')
    
    plt.xlabel("Training Steps", fontsize=11)
    plt.ylabel("Average Episodic Reward", fontsize=11)
    plt.tick_params(axis='both', which='major', labelsize=10)
    
    plt.xlim(min(steps), max(steps)) # Corta o espaço branco lateral
    plt.margins(x=0)
    
    # Pega os limites e marcações (ticks) originais calculados pelo gráfico
    ax = plt.gca()
    y_ticks = ax.get_yticks()
    y_min, y_max = ax.get_ylim()
    
    # Trava os números do eixo Y para que não apareçam no espaço vazio
    ax.set_yticks(y_ticks)
    
    # Aumenta o limite superior em 20% para criar o "teto falso" para a legenda
    plt.ylim(y_min, y_max + (y_max - y_min) * 0.10)
    
    plt.legend(loc='upper right', fontsize=8, frameon=True, framealpha=0.9, edgecolor='lightgrey')
    
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    os.makedirs("results", exist_ok=True)
    
    plt.savefig("results/Average_Reward.pdf", dpi=300, bbox_inches='tight', pad_inches=0.02)
    plt.close()
    print("✅ Gráfico de Treinamento salvo padronizado!")

if __name__ == "__main__":
    base_log_dir = r"C:\Users\adriely\Desktop\SGHaul_DRL_Project_Testes\logs\ppo_sghaul_tensorboard"
    arquivos_tb = glob.glob(os.path.join(base_log_dir, "**", "events.out.tfevents.*"), recursive=True)
    
    if arquivos_tb:
        caminho_mais_recente = max(arquivos_tb, key=os.path.getctime)
        print(f"Lendo automaticamente o último log: {caminho_mais_recente}")
        plot_learning_curve_tb(caminho_mais_recente)
    else:
        print("Nenhum log encontrado!")

Lendo automaticamente o último log: C:\Users\adriely\Desktop\SGHaul_DRL_Project_Testes\logs\ppo_sghaul_tensorboard\PPO_12\events.out.tfevents.1786737570.MC618396.10404.0
Gerando gráfico de Treinamento...
✅ Gráfico de Treinamento salvo padronizado!


In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from envs.sghaul_env import SGHaulEnv

def gerar_graficos_finais_artigo():
    # Trava da Semente
    seed_val = 83604
    os.environ['PYTHONHASHSEED'] = str(seed_val)
    random.seed(seed_val)
    np.random.seed(seed_val)

    print("Gerando gráficos definitivos para a janela de estresse (55s - 65s)...")
    env = SGHaulEnv()
    caminho_modelo = r"C:\Users\adriely\Desktop\SGHaul_DRL_Project_Testes\models\ppo_sghaul_slicing.zip"
    model = PPO.load(caminho_modelo, env=env)

    obs, info = env.reset(seed=seed_val) 
    
    tempos, alocacao_urllc, alocacao_embb = [], [], []
    latencia_urllc, vazao_urllc, vazao_embb = [], [], []

    passo = 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        
        tempo_atual = passo * 0.01
        tempos.append(tempo_atual)
        
        fatia_urllc = np.clip(float(action[0]), 0.0, 1.0)
        fatia_embb = 1.0 - fatia_urllc
        alocacao_urllc.append(fatia_urllc * 100)
        alocacao_embb.append(fatia_embb * 100)
            
        latencia_urllc.append(info.get('avg_latency_uc16', 0.0))
        vazao_urllc.append(info.get('throughput_urllc_mbps', 0.0) * 1000.0)
        vazao_embb.append(info.get('throughput_mbps', 0.0) * 1000.0)

        if terminated or truncated:
            break
            
        passo += 1

    # ==========================================
    # RECORTANDO A JANELA (55s a 65s) E PLOTANDO
    # ==========================================
    janela_inicio = 55.0
    janela_fim = 65.0
    
    mascara = (np.array(tempos) >= janela_inicio) & (np.array(tempos) <= janela_fim)
    tempos_foco = np.array(tempos)[mascara]
    
    vazao_urllc_foco = pd.Series(np.array(vazao_urllc)[mascara]).rolling(window=10, min_periods=1).mean()
    vazao_embb_foco = pd.Series(np.array(vazao_embb)[mascara]).rolling(window=10, min_periods=1).mean()
    
    aloc_urllc_foco = pd.Series(np.array(alocacao_urllc)[mascara]).rolling(window=10, min_periods=1).mean()
    aloc_embb_foco = pd.Series(np.array(alocacao_embb)[mascara]).rolling(window=10, min_periods=1).mean()
    
    latencia_urllc_foco = pd.Series(np.array(latencia_urllc)[mascara]).rolling(window=10, min_periods=1).mean()

    plt.rcParams['font.family'] = 'serif'
    os.makedirs("results", exist_ok=True)
    
    # 1. Gráfico Final de Alocação
    plt.figure(figsize=(7, 4)) 
    plt.plot(tempos_foco, aloc_urllc_foco, label='URLLC Slice (%)', color='red', linewidth=1.5)
    plt.plot(tempos_foco, aloc_embb_foco, label='eMBB Slice (%)', color='blue', linewidth=1.5)
    plt.fill_between(tempos_foco, aloc_urllc_foco, color='red', alpha=0.1)
    plt.fill_between(tempos_foco, aloc_embb_foco, color='blue', alpha=0.1)
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Bandwidth Allocation (%)", fontsize=11)
    plt.xlim(janela_inicio, janela_fim) 
    
    # Sobe o limite para 115, mas trava as marcações em 100
    plt.ylim(0, 120) 
    plt.yticks([0, 20, 40, 60, 80, 100])
    plt.margins(x=0)
    
    plt.legend(loc='upper right', fontsize=8, frameon=True, framealpha=0.9, edgecolor='lightgrey')
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/final_allocation_paper.pdf', dpi=300, bbox_inches='tight', pad_inches=0.02)
    plt.close()

    # 2. Gráfico Final de Vazão
    plt.figure(figsize=(7, 4))
    plt.plot(tempos_foco, vazao_urllc_foco, label='URLLC Throughput', color='red', linewidth=2, zorder=1)
    plt.fill_between(tempos_foco, vazao_urllc_foco, color='red', alpha=0.2, zorder=1)
    plt.plot(tempos_foco, vazao_embb_foco, label='eMBB Throughput', color='blue', linewidth=2, zorder=2)
    plt.fill_between(tempos_foco, vazao_embb_foco, color='blue', alpha=0.3, zorder=2)
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Throughput (Kbps)", fontsize=11)
    plt.xlim(janela_inicio, janela_fim)
    plt.ylim(bottom=-0.5) 
    plt.margins(x=0)
    
    # Legenda no topo esquerdo e fonte reduzida
    plt.legend(loc='upper right', fontsize=8, frameon=True, framealpha=0.9, edgecolor='lightgrey')
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/final_throughput_paper.pdf', dpi=300, bbox_inches='tight', pad_inches=0.02)
    plt.close()
    
    # 3. Gráfico Final de Latência
    plt.figure(figsize=(7, 4))
    plt.plot(tempos_foco, latencia_urllc_foco, color='red', linewidth=1.5, alpha=0.8, label='URLLC Packet Latency')
    plt.axhline(y=300, color='black', linestyle='--', linewidth=1.5, label='Delay Bound (300ms)')
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Delay (ms)", fontsize=11)
    plt.xlim(janela_inicio, janela_fim)
    plt.ylim(-10, 400) 
    plt.margins(x=0)
    
    # Legenda no topo esquerdo e fonte reduzida
    plt.legend(loc='upper right', fontsize=8, frameon=True, framealpha=0.9, edgecolor='lightgrey')
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/final_latency_paper.pdf', dpi=300, bbox_inches='tight', pad_inches=0.02)
    plt.close()

    print("✅ Gráficos DEFINITIVOS com legendas ajustadas no topo esquerdo!")

if __name__ == "__main__":
    gerar_graficos_finais_artigo()

Gerando gráficos definitivos para a janela de estresse (55s - 65s)...
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
✅ Gráficos DEFINITIVOS com legendas ajustadas no topo esquerdo!


# testes abaixo!!   

In [17]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from envs.sghaul_env import SGHaulEnv

def gerar_graficos_temporais():
    # ==========================================
    # TRAVA GLOBAL DE SEMENTE (A SUA SEMENTE CRÍTICA!)
    # ==========================================
    seed_val = 83604
    os.environ['PYTHONHASHSEED'] = str(seed_val)
    random.seed(seed_val)
    np.random.seed(seed_val)
    # ==========================================

    print("Iniciando simulação temporal da IA...")
    env = SGHaulEnv()
    caminho_modelo = r"C:\Users\adriely\Desktop\SGHaul_DRL_Project_Testes\models\ppo_sghaul_slicing.zip"
    model = PPO.load(caminho_modelo, env=env)

    obs, info = env.reset(seed=seed_val) 
    
    tempos, alocacao_urllc, alocacao_embb = [], [], []
    latencia_urllc, vazao_urllc, vazao_embb = [], [], []

    # AUMENTAMOS PARA 2000 PASSOS (20 SEGUNDOS) PARA FUGIR DO INÍCIO
    passos_simulacao = 2000 

    for passo in range(passos_simulacao):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        
        # --- DEBUG 1: VERIFICAR CHAVES DO INFO NO PRIMEIRO PASSO ---
        if passo == 0:
            print("\n[DEBUG] Chaves disponíveis no 'info':", info.keys())
        
        # 1. TEMPO
        tempo_atual = passo * 0.01
        tempos.append(tempo_atual)
        
        # 2. ALOCAÇÃO
        fatia_urllc = np.clip(float(action[0]), 0.0, 1.0)
        fatia_embb = 1.0 - fatia_urllc
        
        alocacao_urllc.append(fatia_urllc * 100)
        alocacao_embb.append(fatia_embb * 100)
            
        latencia_urllc.append(info.get('avg_latency_uc16', 0.0))
        vazao_urllc.append(info.get('throughput_urllc_mbps', 0.0) * 1000.0)
        
        # Atenção aqui: se a chave estiver errada, vai continuar retornando 0.0
        vazao_embb.append(info.get('throughput_mbps', 0.0) * 1000.0)

        if terminated or truncated:
            break

    # ==========================================
    # --- DEBUG 2: RAIO-X DOS DADOS ANTES DO GRÁFICO ---
    # ==========================================
    print(f"\n[DEBUG] Vazão URLLC -> Média: {np.mean(vazao_urllc):.2f} Kbps | Máxima: {np.max(vazao_urllc):.2f} Kbps")
    print(f"[DEBUG] Vazão eMBB  -> Média: {np.mean(vazao_embb):.2f} Kbps | Máxima: {np.max(vazao_embb):.2f} Kbps")
    
    if np.max(vazao_embb) == 0.0:
        print("[ALERTA CRÍTICO] A vazão máxima do eMBB foi ZERO absoluto. Ou a chave 'throughput_mbps' está errada, ou nenhum pacote foi gerado na simulação toda.")

    # 3. SUAVIZAÇÃO PARA OS GRÁFICOS
    vazao_urllc_suave = pd.Series(vazao_urllc).rolling(window=10, min_periods=1).mean()
    vazao_embb_suave = pd.Series(vazao_embb).rolling(window=10, min_periods=1).mean()
    
    aloc_urllc_suave = pd.Series(alocacao_urllc).rolling(window=10, min_periods=1).mean()
    aloc_embb_suave = pd.Series(alocacao_embb).rolling(window=10, min_periods=1).mean()

    plt.rcParams['font.family'] = 'serif'
    os.makedirs("results", exist_ok=True)
    
    # ==========================================
    # --- DEBUG 3: CORTANDO O GRÁFICO (Ex: do segundo 10 ao 20) ---
    # ==========================================
    # Vamos criar uma máscara para focar apenas de 10s até 20s
    mascara = (np.array(tempos) >= 10.0) & (np.array(tempos) <= 20.0)
    tempos_foco = np.array(tempos)[mascara]
    
    # 1. Gráfico de Alocação (Focado)
    plt.figure(figsize=(7, 4)) 
    plt.plot(tempos_foco, np.array(aloc_urllc_suave)[mascara], label='URLLC Slice (%)', color='red', linewidth=1.5)
    plt.plot(tempos_foco, np.array(aloc_embb_suave)[mascara], label='eMBB Slice (%)', color='blue', linewidth=1.5)
    plt.fill_between(tempos_foco, np.array(aloc_urllc_suave)[mascara], color='red', alpha=0.1)
    plt.fill_between(tempos_foco, np.array(aloc_embb_suave)[mascara], color='blue', alpha=0.1)
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Bandwidth Allocation (%)", fontsize=11)
    plt.ylim(0, 100) 
    plt.tick_params(axis='both', which='major', labelsize=10)
    plt.legend(loc='center right', fontsize=10)
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/temporal_allocation_debug.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # 3. Gráfico de Vazão (Focado)
    plt.figure(figsize=(7, 4))
    plt.plot(tempos_foco, np.array(vazao_urllc_suave)[mascara], label='URLLC Throughput', color='red', linewidth=2, zorder=1)
    plt.fill_between(tempos_foco, np.array(vazao_urllc_suave)[mascara], color='red', alpha=0.2, zorder=1)
    plt.plot(tempos_foco, np.array(vazao_embb_suave)[mascara], label='eMBB Throughput', color='blue', linewidth=2, zorder=2)
    plt.fill_between(tempos_foco, np.array(vazao_embb_suave)[mascara], color='blue', alpha=0.3, zorder=2)
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Throughput (Kbps)", fontsize=11)
    plt.ylim(bottom=-0.5) 
    plt.tick_params(axis='both', which='major', labelsize=10)
    plt.legend(loc='upper right', fontsize=10)
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/temporal_throughput_debug.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    print("\n✅ Gráficos temporais de DEBUG gerados com sucesso!")

if __name__ == "__main__":
    gerar_graficos_temporais()

Iniciando simulação temporal da IA...
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

[DEBUG] Chaves disponíveis no 'info': dict_keys(['throughput_mbps', 'throughput_urllc_mbps', 'demanda_urllc_mbps', 'demanda_embb_mbps', 'alocado_urllc_mbps', 'alocado_embb_mbps', 'fila_urllc_kb', 'fila_embb_kb', 'drops', 'topologia', 'violacoes_uc16', 'total_pacotes_uc16', 'avg_latency_uc16', 'violacoes_uc3', 'total_pacotes_uc3', 'avg_latency_uc3'])

[DEBUG] Vazão URLLC -> Média: 87.19 Kbps | Máxima: 409.60 Kbps
[DEBUG] Vazão eMBB  -> Média: 0.00 Kbps | Máxima: 0.00 Kbps
[ALERTA CRÍTICO] A vazão máxima do eMBB foi ZERO absoluto. Ou a chave 'throughput_mbps' está errada, ou nenhum pacote foi gerado na simulação toda.

✅ Gráficos temporais de DEBUG gerados com sucesso!


In [18]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from envs.sghaul_env import SGHaulEnv

def gerar_graficos_visao_macro():
    # Trava da Semente
    seed_val = 83604
    os.environ['PYTHONHASHSEED'] = str(seed_val)
    random.seed(seed_val)
    np.random.seed(seed_val)

    print("Iniciando varredura MACRO do episódio completo...")
    env = SGHaulEnv()
    caminho_modelo = r"C:\Users\adriely\Desktop\SGHaul_DRL_Project_Testes\models\ppo_sghaul_slicing.zip"
    model = PPO.load(caminho_modelo, env=env)

    obs, info = env.reset(seed=seed_val) 
    
    tempos, alocacao_urllc, alocacao_embb = [], [], []
    latencia_urllc, vazao_urllc, vazao_embb = [], [], []

    passo = 0
    # Rodando até o ambiente avisar que o episódio acabou
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        
        tempo_atual = passo * 0.01
        tempos.append(tempo_atual)
        
        fatia_urllc = np.clip(float(action[0]), 0.0, 1.0)
        fatia_embb = 1.0 - fatia_urllc
        alocacao_urllc.append(fatia_urllc * 100)
        alocacao_embb.append(fatia_embb * 100)
            
        latencia_urllc.append(info.get('avg_latency_uc16', 0.0))
        vazao_urllc.append(info.get('throughput_urllc_mbps', 0.0) * 1000.0)
        vazao_embb.append(info.get('throughput_mbps', 0.0) * 1000.0)

        if terminated or truncated:
            print(f"\n[INFO] Episódio finalizado naturalmente no passo {passo} (Tempo total: {tempo_atual:.2f}s)")
            break
            
        passo += 1

    # ==========================================
    # --- RADAR DE TRÁFEGO eMBB ---
    # ==========================================
    print("\n" + "="*50)
    print("🔬 RELATÓRIO DO RADAR DE TRÁFEGO eMBB")
    print("="*50)
    
    max_embb = np.max(vazao_embb)
    print(f"Vazão Máxima Atingida: {max_embb:.2f} Kbps")
    
    if max_embb > 0:
        idx_pico = np.argmax(vazao_embb)
        tempo_pico = tempos[idx_pico]
        print(f"🌟 MOMENTO DE OURO: O pico absoluto ocorreu exatamente no segundo {tempo_pico:.2f}s")
        print(f"   -> Sugestão de Janela para o Artigo: Do segundo {max(0, tempo_pico-5):.2f}s até {tempo_pico+5:.2f}s")
        
        # Encontra todos os momentos onde a vazão foi considerável (ex: > 1 Kbps)
        momentos_ativos = [t for t, v in zip(tempos, vazao_embb) if v > 1.0]
        if momentos_ativos:
            print(f"   -> Tráfego ativo detectado pela primeira vez em: {momentos_ativos[0]:.2f}s")
            print(f"   -> Tráfego ativo detectado pela última vez em: {momentos_ativos[-1]:.2f}s")
    else:
        print("❌ ALERTA: A semente 83604 não gerou absolutamente NENHUM pacote eMBB durante o episódio INTEIRO.")
        print("   -> Solução: Você precisará trocar a 'seed_val' no início do script para encontrar uma semente viável.")
    print("="*50 + "\n")

    # 3. SUAVIZAÇÃO PARA OS GRÁFICOS
    vazao_urllc_suave = pd.Series(vazao_urllc).rolling(window=10, min_periods=1).mean()
    vazao_embb_suave = pd.Series(vazao_embb).rolling(window=10, min_periods=1).mean()
    aloc_urllc_suave = pd.Series(alocacao_urllc).rolling(window=10, min_periods=1).mean()
    aloc_embb_suave = pd.Series(alocacao_embb).rolling(window=10, min_periods=1).mean()

    plt.rcParams['font.family'] = 'serif'
    os.makedirs("results", exist_ok=True)
    
    # Gráfico Macro de Alocação
    plt.figure(figsize=(10, 4)) 
    plt.plot(tempos, aloc_urllc_suave, label='URLLC Slice (%)', color='red', linewidth=1.0)
    plt.plot(tempos, aloc_embb_suave, label='eMBB Slice (%)', color='blue', linewidth=1.0)
    plt.fill_between(tempos, aloc_urllc_suave, color='red', alpha=0.1)
    plt.fill_between(tempos, aloc_embb_suave, color='blue', alpha=0.1)
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Bandwidth Allocation (%)", fontsize=11)
    plt.ylim(0, 100) 
    plt.legend(loc='center right', fontsize=10)
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/temporal_allocation_macro.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Gráfico Macro de Vazão
    plt.figure(figsize=(10, 4))
    plt.plot(tempos, vazao_urllc_suave, label='URLLC Throughput', color='red', linewidth=1.5, zorder=1)
    plt.fill_between(tempos, vazao_urllc_suave, color='red', alpha=0.2, zorder=1)
    plt.plot(tempos, vazao_embb_suave, label='eMBB Throughput', color='blue', linewidth=1.5, zorder=2)
    plt.fill_between(tempos, vazao_embb_suave, color='blue', alpha=0.3, zorder=2)
    plt.xlabel("Simulation Time (s)", fontsize=11)
    plt.ylabel("Throughput (Kbps)", fontsize=11)
    plt.ylim(bottom=-0.5) 
    plt.legend(loc='upper right', fontsize=10)
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('results/temporal_throughput_macro.pdf', dpi=300, bbox_inches='tight')
    plt.close()

if __name__ == "__main__":
    gerar_graficos_visao_macro()

Iniciando varredura MACRO do episódio completo...
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

[INFO] Episódio finalizado naturalmente no passo 9999 (Tempo total: 99.99s)

🔬 RELATÓRIO DO RADAR DE TRÁFEGO eMBB
Vazão Máxima Atingida: 512.00 Kbps
🌟 MOMENTO DE OURO: O pico absoluto ocorreu exatamente no segundo 61.37s
   -> Sugestão de Janela para o Artigo: Do segundo 56.37s até 66.37s
   -> Tráfego ativo detectado pela primeira vez em: 30.02s
   -> Tráfego ativo detectado pela última vez em: 92.63s

